In [ ]:
from src.common.constants import Constants as consts
from src.models.model_trainer import ModelTrainer
from src.models.logistic_regression_clf import LogisticRegressionClf, objective_acc
from src.common.constants import ExperimentPreprocessConfig
from src.preprocessing.experiment_preprocessor import ExperimentPreprocessor


FRACTION = 0.5
IS_AUDIO_IDS_SAMPLING = False
FILE_SUFFIX = consts.wavlm_emb_suffix
N_TRIALS = 50
MODEL = LogisticRegressionClf
OBJECTIVE = objective_acc

In [ ]:
exp_preprocessor = ExperimentPreprocessor(
    load_file_name=consts.feature_extracted,
    save_file_name=consts.feature_extracted,
    feat_suffix=FILE_SUFFIX
)

balance_strategy_ratios = {
    "unbalanced": [None],
    "oversample": [0.5, 0.75, 1.0],
    "undersample": [0.5, 0.75, 1.0],
    "mix": [[0.5, 0.75], [0.5, 1.0], [0.75, 1.0]],
}

trainer = ModelTrainer()

2026-03-20 19:40:10 | INFO     | FeatureLoader | Using WavLM embeddings suffix
2026-03-20 19:40:10 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted_wavlm.npy
2026-03-20 19:40:10 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted.csv
2026-03-20 19:40:10 | INFO     | Collector | Using WavLM embeddings suffix
2026-03-20 19:40:10 | INFO     | Collector | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted.csv
2026-03-20 19:40:10 | INFO     | Collector | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted_wavlm.npy


In [ ]:
def create_comparision_dict(balance_strategy_ratios):
    comparison_dict = {}
    for strategy, ratios in balance_strategy_ratios.items():
        comparison_dict[f"{strategy}:{ratios}"] = ratios
    return comparison_dict

def train_all_balancers(
    model,
    objective,
    n_trials: int,
    exp_preprocessor: ExperimentPreprocessor,
    balance_strategy_ratios: dict[str, list[float] | list[list[float]]],
):
    comparison_dict = create_comparision_dict(balance_strategy_ratios)

    for strategy, ratios in balance_strategy_ratios.items():
        for ratio in ratios:
            preprocessing_config = ExperimentPreprocessConfig(
                splits_names=["train", "dev"],
                fraction=FRACTION,
                is_audio_ids_sampling=IS_AUDIO_IDS_SAMPLING,
                balance_splits_strategy=[strategy, ratio],
            )

            data_for_exp = exp_preprocessor.preprocess_data(**preprocessing_config.get_dict())
            meta_train, X_train = data_for_exp["train"]
            y_train = trainer.get_target(metadata=meta_train)
            meta_dev, X_dev = data_for_exp["dev"]
            y_dev = trainer.get_target(metadata=meta_dev)

            trainer.optuna_train(
                model=model,
                objective=objective,
                n_trials=n_trials,
                # max_iter=100,
                X_train=X_train,
                y_train=y_train,
                X_dev=X_dev,
                y_dev=y_dev,
            )

            best_value = trainer.get_best_value()
            comparison_dict_key = f"{strategy}:{ratio}"
            comparison_dict[comparison_dict_key] = best_value

    return comparison_dict


results = train_all_balancers(
    model=MODEL,
    objective=OBJECTIVE,
    n_trials=N_TRIALS,
    exp_preprocessor=exp_preprocessor,
    balance_strategy_ratios=balance_strategy_ratios,
)
print(results)